# EX_10 — LangGraph y flujos (ejercicios)

**Notebook de referencia:** `notebook/10_LangGraph_Flujos.ipynb`

**Tiempo orientativo:** ~30 minutos.


## Actividad 1 — Estado TypedDict

Define un `TypedDict` de estado con al menos: `question: str`, `answer: str`, `step_count: int`.


In [2]:
# =====================================================================
# ACTIVIDAD 1: DEFINICIÓN DEL ESTADO CON TYPEDDICT
# =====================================================================

from typing import TypedDict

# Definimos la clase del estado heredando de TypedDict
class QAState(TypedDict):
    """
    Representa el estado global que se comparte entre los diferentes
    nodos de un grafo de ejecución (ej. en LangGraph).
    """
    question: str    # Almacena la pregunta inicial del usuario
    answer: str      # Guardará la respuesta final generada por el sistema
    step_count: int  # Contador de pasos para mitigar bucles infinitos


# # Ejemplo opcional de cómo se crearía una instancia de este estado:
# estado_inicial: QAState = {
#     "question": "How many words in this sentence?",
#     "answer": "",
#     "step_count": 0
# }


## Actividad 2 — Dos nodos

Esquematiza (pseudocódigo) nodos `retrieve` y `generate` que incrementen `step_count`. No hace falta ejecutar LangGraph si aún no está importado en el entorno.


In [ ]:
# def retrieve(state):
#     ...
# def generate(state):
#     ...


In [3]:
# =====================================================================
# ACTIVIDAD 2: ESQUEMA DE NODOS RETRIEVE Y GENERATE (PSEUDOCÓDIGO)
# =====================================================================

from typing import TypedDict

# Recordamos la estructura del estado de la Actividad 1
class QAState(TypedDict):
    question: str
    answer: str
    step_count: int

# ---------------------------------------------------------------------
# NODO 1: RETRIEVE (Recuperación de Información)
# ---------------------------------------------------------------------
def retrieve(state: QAState) -> dict:
    """
    Primer nodo del flujo. Toma la pregunta del estado, simula la
    búsqueda en una base de conocimientos y actualiza el contador de pasos.
    """
    print(f"--- NODO: RETRIEVE (Paso actual: {state['step_count']}) ---")

    # 1. Accedemos a la pregunta guardada en el estado
    pregunta = state["question"]

    # 2. (Simulación) Aquí iría la lógica de búsqueda en VectorDB o diccionario
    # contexto_recuperado = vector_db.similarity_search(pregunta)

    # 3. Retornamos un diccionario con los campos que queremos actualizar en el estado global.
    # Incrementamos 'step_count' en 1 para llevar el control del ciclo.
    return {
        "step_count": state["step_count"] + 1
        # "context": contexto_recuperado (Opcional si guardamos el contexto)
    }

# ---------------------------------------------------------------------
# NODO 2: GENERATE (Generación de Respuesta)
# ---------------------------------------------------------------------
def generate(state: QAState) -> dict:
    """
    Segundo nodo del flujo. Toma el estado actualizado, envía la información
    al LLM para redactar la respuesta y vuelve a incrementar el contador de pasos.
    """
    print(f"--- NODO: GENERATE (Paso actual: {state['step_count']}) ---")

    # 1. Accedemos a los datos que el estado acumuló hasta ahora
    pregunta = state["question"]

    # 2. (Simulación) Aquí el LLM generaría el texto final
    # respuesta_llm = llm.invoke(f"Responde a {pregunta} con el contexto...")
    respuesta_simulada = "Esta es la respuesta generada por el modelo."

    # 3. Retornamos las actualizaciones del estado.
    # Guardamos el texto final en 'answer' e incrementamos de nuevo 'step_count'.
    return {
        "answer": respuesta_simulada,
        "step_count": state["step_count"] + 1
    }

## Actividad 3 — Condicional

Describe en markdown cuándo enrutarías a un nodo `human_review` (p. ej. si `confidence < 0.5`).


_Criterio de enrutamiento:_

...


In [ ]:
### 🔀 Lógica de Enrutamiento Condicional: Nodo `human_review`

En arquitecturas avanzadas de agentes y flujos de trabajo basados en grafos (como LangGraph), un nodo de **revisión humana (`human_review`)** actúa como una red de seguridad indispensable.

El enrutamiento hacia este nodo se gestiona mediante una **función de borde condicional (Conditional Edge)** que analiza el estado actual del flujo justo después de la evaluación del modelo o del auditor.

---

#### 📋 Criterios Clave para Enrutar a `human_review`

El sistema desviará el control hacia un operador humano en los siguientes escenarios:

##### 1. Confianza del Clasificador/LLM por debajo del Umbral (`confidence < 0.5`)
* **Cuándo ocurre:** El modelo clasifica una pregunta del usuario (por ejemplo, para asignarla a un departamento o categoría del manual) pero el score de probabilidad o confianza devuelto es inferior al 50%.
* **Acción:** En lugar de arriesgarse a dar una respuesta incorrecta o ejecutar una herramienta equivocada, el flujo se detiene y solicita validación humana.

##### 2. Fallo Reiterado en el Control de Calidad (`step_count > max_intentos`)
* **Cuándo ocurre:** Si el nodo de autocorrecación ha intentado reescribir la respuesta dos o tres veces, pero el nodo de *Audit* o *Critic* sigue marcando la respuesta como desaprobada (debido a alucinaciones o falta de contexto).
* **Acción:** Para evitar un bucle infinito que consuma créditos de la API, se rompe el ciclo y se escala el caso al equipo de soporte humano con el historial del fallo.

##### 3. Detección de Temas Sensibles, Críticos o Inseguros
* **Cuándo ocurre:** El clasificador de entrada detecta palabras clave relacionadas con solicitudes legales, cancelaciones de cuentas VIP, quejas graves de clientes, datos financieros sensibles o contenido potencialmente peligroso.
* **Acción:** Por cumplimiento normativo y seguridad empresarial (*Guardrails*), estas intenciones nunca se responden de forma 100% automatizada.

---

#### 🛠️ Ejemplo de Lógica en Pseudocódigo

En el grafo, el enrutador condicional implementaría una lógica similar a esta:

```python
def routing_logic(state: QAState):
    # Criterio 1: Baja confianza
    if state.get("confidence") < 0.5:
        return "human_review"

    # Criterio 2: Límite de intentos de corrección alcanzado
    if state.get("step_count") >= 3 and not state.get("quality_approved"):
        return "human_review"

    # Si todo está bien, continúa al cierre o respuesta final
    return "finalize"